# Predicate benchmarks: expressions, ordinary UDFs and Menudet

Three float32 predicates over compressed 2-D arrays. Public kernels live in [compute_expr_udf.py](compute_expr_udf.py). Numba/Cython remain ordinary `lazyudf` baselines; Menudet kernels bind directly, never through `lazyudf`.

[Historical legacy DSL measurements](../../plans/compute-expr-udf-legacy.md) are preserved separately. Their backend, threads, versions and correctness were not recorded: size-matched ratios below are **historical context, not proof of a regression**. All current competitors are rerun.

In [1]:
# SPDX-License-Identifier: BSD-3-Clause
# Copyright (c) 2019-present, Blosc Development Team
import json
import os
import platform
from pathlib import Path
from time import perf_counter

import numba as nb
import numexpr as ne
import numpy as np
import blosc2
# Find the shared module when Jupyter starts in the repository or notebook directory.
import sys
for directory in (Path.cwd(), *Path.cwd().parents):
    if (directory / 'bench/ndarray/compute_expr_udf.py').is_file():
        sys.path.insert(0, str(directory))
        break
from bench.ndarray.compute_expr_udf import EXPRESSIONS, KERNELS, reference, qualified_measure, timing_label, execution_report

%load_ext cython

In [2]:
# Environment overrides allow a small smoke run without editing the notebook.
side = int(os.environ.get('BLOSC2_BENCH_SIDE', '10000'))
shape = (side, side)
repeat = int(os.environ.get('BLOSC2_BENCH_REPEAT', '3'))
threads = int(os.environ.get('BLOSC2_BENCH_THREADS', str(blosc2.nthreads)))
assert min(side, repeat, threads) > 0
blosc2.set_nthreads(threads)
nb.set_num_threads(threads)
ne.set_num_threads(threads)
# Explicit settings replace the old notebook's mutable global compression defaults.
cparams = blosc2.CParams(codec=blosc2.Codec.LZ4, clevel=5, nthreads=threads)
# Match the historical grid at the full size; use partial tails in smoke runs.
chunks = (min(1250, side), side)
blocks = (min(2, side), side)
x = blosc2.linspace(0, 1, side * side, shape=shape, dtype=np.float32, chunks=chunks, blocks=blocks, cparams=cparams)
y = blosc2.linspace(-1, 1, side * side, shape=shape, dtype=np.float32, chunks=chunks, blocks=blocks, cparams=cparams)
z = blosc2.linspace(0, 10, side * side, shape=shape, dtype=np.float32, chunks=chunks, blocks=blocks, cparams=cparams)
compressed = (x, y, z)
resident = tuple(value[:] for value in compressed)
storage = dict(chunks=chunks, blocks=blocks, cparams=cparams)
metadata = dict(python=platform.python_version(), platform=platform.platform(), processor=platform.processor(),
                package=blosc2.__file__, extension=blosc2.blosc2_ext.__file__, blosc2=blosc2.__version__,
                c_blosc2=blosc2.blosclib_version, numpy=np.__version__, numba=nb.__version__, numexpr=ne.__version__,
                shape=shape, dtype='float32', chunks=chunks, blocks=blocks, cparams=repr(cparams), threads=threads,
                repeat=repeat, cython_openmp=False, input_layout='C-order linspaces x=[0,1], y=[-1,1], z=[0,10]; no seed',
                protocol='validate first result; best of warm repeats; compute allocates/compresses NDArray, getitem allocates NumPy; no conversion in timing',
                environment={k:v for k,v in os.environ.items() if k.startswith(('ME_', 'BLOSC_ME_')) or k in ('CC','CFLAGS')})
print(f'{shape}, float32, {threads} threads, {repeat} warm repetitions')

(10000, 10000), float32, 12 threads, 3 warm repetitions


In [3]:
@nb.njit(parallel=True)
def udf_numba(inputs, output, offset):
    x = inputs[0]
    if len(inputs) == 1:
        for i in nb.prange(x.shape[0]):
            for j in range(x.shape[1]):
                output[i,j] = x[i,j] < 0.5
    elif len(inputs) == 2:
        y = inputs[1]
        for i in nb.prange(x.shape[0]):
            for j in range(x.shape[1]):
                output[i,j] = x[i,j]**2 + y[i,j]**2 <= 2*x[i,j]*y[i,j] + 1
    else:
        y, z = inputs[1], inputs[2]
        for i in nb.prange(x.shape[0]):
            for j in range(x.shape[1]):
                output[i,j] = (np.sin(x[i,j])**3 + np.cos(y[i,j])**2) >= (np.cos(x[i,j])*np.sin(y[i,j]) + z[i,j])

In [4]:
%%cython
# As in the historical notebook, no OpenMP compile/link flags are enabled.
import numpy as np
cimport numpy as np
cimport cython
from cython.parallel cimport prange
from libc.math cimport sinf, cosf

@cython.boundscheck(False)
@cython.wraparound(False)
def udf_cython(inputs, np.npy_bool[:, ::1] output, offset):
    cdef const np.npy_float32[:, ::1] x = inputs[0]
    cdef const np.npy_float32[:, ::1] y, z
    cdef Py_ssize_t i, j
    if len(inputs) == 1:
        for i in prange(x.shape[0], nogil=True):
            for j in range(x.shape[1]):
                output[i,j] = x[i,j] < 0.5
    elif len(inputs) == 2:
        y = inputs[1]
        for i in prange(x.shape[0], nogil=True):
            for j in range(x.shape[1]):
                output[i,j] = x[i,j]**2 + y[i,j]**2 <= 2*x[i,j]*y[i,j] + 1
    else:
        y, z = inputs[1], inputs[2]
        for i in prange(x.shape[0], nogil=True):
            for j in range(x.shape[1]):
                output[i,j] = (sinf(x[i,j])**3 + cosf(y[i,j])**2) >= (cosf(x[i,j])*sinf(y[i,j]) + z[i,j])

## Correctness first, then warm timing

Every route checks boolean dtype, shape, and exact predicate values against NumPy before warm timing. Failed routes are labeled UNQUALIFIED, retain diagnostics in JSON, and have no warm times or plot bars. No approximate boolean comparisons are used. Compute and getitem are distinct output contracts, shown in separate plots. Direct Numba/NumExpr reuse an output buffer and are labeled accordingly.

First execution is reported separately, not claimed to be cache-isolated cold compilation. Menudet specialization (including any compilation) is separately timed; decorator processing/import and Cython cell compilation are outside timings. Both TCC and CC are requested explicitly; JIT availability and actual execution-route reports are saved, not inferred from speed.

In [5]:
results = []
policies = {}
for case, expr in enumerate(EXPRESSIONS):
    arrays = compressed[:case+1]
    values = resident[:case+1]
    expected = reference(case, values)
    names = ('x', 'y', 'z')[:case+1]
    operands = dict(zip(names, arrays))
    npoperands = dict(zip(names, values))
    output = np.empty(shape, dtype=np.bool_)
    lazyexpr = blosc2.lazyexpr(expr, operands)
    numba_udf = blosc2.lazyudf(udf_numba, arrays, np.bool_)
    cython_udf = blosc2.lazyudf(udf_cython, arrays, np.bool_)
    operations = {
        'NumPy': (lambda: reference(case, values), False),
        'NumExpr (out)': (lambda: ne.evaluate(expr, local_dict=npoperands, out=output), False),
        'Numba (out)': (lambda: (udf_numba(values, output, None), output)[1], False),
        'LazyExpr compute': (lambda: lazyexpr.compute(**storage), True),
        'LazyExpr getitem': (lambda: lazyexpr[:], False),
        'Numba UDF compute': (lambda: numba_udf.compute(**storage), True),
        'Numba UDF getitem': (lambda: numba_udf[:], False),
        'Cython UDF compute': (lambda: cython_udf.compute(**storage), True),
        'Cython UDF getitem': (lambda: cython_udf[:], False),
    }
    print(f'Predicate {case+1}: {expr}', flush=True)
    for method, (operation, materialized) in operations.items():
        timing = qualified_measure(operation, expected, materialized=materialized, repeat=repeat)
        results.append(dict(case=case+1, method=method, **timing))
        print(f'  {method:23s} {timing_label(timing)}', flush=True)
    for compiler in ('tcc', 'cc'):
        with blosc2.jit_options(jit=True, jit_backend=compiler):
            policies[compiler] = {k:str(v) for k,v in blosc2.jit_config.execution_options().items()}
            start = perf_counter()
            kernel = KERNELS[case].specialize(**operands)
            preparation = perf_counter() - start
            assert kernel.has_jit, f'{compiler}: required benchmark JIT unavailable'
            lazy = kernel.lazy(operands)
            for mode in ('compute', 'getitem'):
                operation = (lambda: lazy.compute(**storage)) if mode == 'compute' else (lambda: lazy[:])
                timing = qualified_measure(operation, expected, materialized=mode == 'compute', repeat=repeat)
                method = f'Menudet {compiler} {mode}'
                results.append(dict(case=case+1, method=method, preparation_s=preparation, has_jit=kernel.has_jit,
                                    execution_report=execution_report(lazy), **timing))
                print(f'  {method:23s} {timing_label(timing)}', flush=True)
metadata['execution_policies'] = policies
Path('compute-expr-udf-results.json').write_text(json.dumps(dict(metadata=metadata, results=results), indent=2) + '\n')
print('Saved raw samples and provenance: compute-expr-udf-results.json')

Predicate 1: x < .5


  NumPy                   0.0197 s


  NumExpr (out)           0.0066 s


  Numba (out)             0.0024 s


  LazyExpr compute        0.0928 s


  LazyExpr getitem        0.0210 s


... (33 outputs left out)

  Menudet tcc getitem     0.1974 s


  Menudet cc compute      0.0860 s


  Menudet cc getitem      0.1110 s


Saved raw samples and provenance: compute-expr-udf-results.json


In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=('NDArray: allocate + compress', 'NumPy: allocate values'))
for column, mode in enumerate(('compute', 'getitem'), 1):
    methods = [f'{method} {mode}' for method in ('LazyExpr', 'Numba UDF', 'Cython UDF', 'Menudet tcc', 'Menudet cc')]
    for method in methods:
        times = [next(row['best_s'] for row in results if row['case'] == case and row['method'] == method) for case in (1,2,3)]
        fig.add_trace(go.Bar(name=method, x=['predicate 1','predicate 2','predicate 3'], y=times), row=1, col=column)
fig.update_layout(title=f'Current predicate benchmarks — {side*side:,} float32 elements, {threads} threads (failed checks omitted)',
                  barmode='group', template='plotly_white', height=550)
fig.update_yaxes(title_text='Warm best time (s); lower is better')
fig.show(renderer='plotly_mimetype')

In [7]:
# Historical numbers stay explicitly labeled, never mixed into the current plot.
legacy = {'compute': (0.069, 0.132, 0.235), 'getitem': (0.079, 0.148, 0.245)}
if shape == (10000, 10000):
    print('Historical context only: new / saved legacy time (not a controlled regression test)')
    for compiler in ('tcc', 'cc'):
        for mode, old_times in legacy.items():
            current = [next(row['best_s'] for row in results if row['case'] == case and row['method'] == f'Menudet {compiler} {mode}') for case in (1,2,3)]
            print(f'{compiler} {mode}: ' + ', '.join(f'{new/old:.2f}x' if new is not None else 'UNQUALIFIED' for new, old in zip(current, old_times)))
else:
    print('Smoke shape: no ratios against historical 100M-element measurements.')

Historical context only: new / saved legacy time (not a controlled regression test)
tcc compute: 0.17x, 0.11x, 0.69x
tcc getitem: 0.17x, 0.12x, 0.81x
cc compute: 0.15x, 0.11x, 0.37x
cc getitem: 0.15x, 0.12x, 0.45x
